In [1]:
import polars as pl

In [5]:
ds = pl.read_parquet('data/ml_labels.parquet')


In [17]:
ds

ticker,signal_date,label,alpha,score,momentum_6m,momentum_3m,momentum_1m,above_200ma,pct_below_52w_high,rsi_14,volume_trend_20d,spy_above_200ma,spy_momentum_6m,gross_margin,net_margin,revenue_growth,net_income_trend,asset_liability_ratio,operating_cf_trend
str,datetime[μs],i64,f64,i64,f64,f64,f64,i64,f64,f64,f64,i64,f64,f64,f64,f64,i64,f64,i64
"""EQIX""",2011-02-14 00:00:00,1,11.066557,3,1.287713,8.904281,4.813359,1,-13.179163,57.89136,61741.729323,1,24.262292,0.446264,0.082669,6.750544,1,null,null
"""INCY""",2011-02-14 00:00:00,1,14.52516,3,13.875969,-8.587431,-7.957396,1,-14.4438,54.576266,-67486.240602,1,24.262292,null,null,null,null,null,null
"""BWA""",2011-02-14 00:00:00,0,-1.826308,2,69.583277,34.18358,12.284019,1,0.0,71.196764,209675.731579,1,24.262292,0.147694,0.016735,-32.229988,0,null,null
"""RL""",2011-02-14 00:00:00,1,20.16445,2,59.341485,22.440515,16.920111,1,-0.086621,89.698496,49784.887218,1,24.262292,0.605303,0.092924,21.689567,1,null,1
"""GT""",2011-02-14 00:00:00,0,6.992893,2,44.298706,41.584177,14.675227,1,-1.174835,74.505863,252754.736842,1,24.262292,0.196579,0.01642,11.209739,1,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""COR""",2025-10-13 00:00:00,0,-5.323125,2,12.266803,8.126329,5.117014,1,0.0,71.531786,-93455.263158,1,23.003772,0.03604,0.008522,365.231157,1,null,1
"""NEE""",2025-10-13 00:00:00,0,4.955757,2,28.664623,12.442498,16.34563,1,-0.821038,87.012191,139408.120301,1,23.003772,null,0.316875,72.739541,1,1.441863,1
"""GILD""",2025-10-13 00:00:00,1,13.633638,2,14.66908,8.1916,3.006334,1,-2.055488,62.374658,63946.766917,1,23.003772,0.788054,0.276758,454.09291,1,null,null


In [10]:

# Group by label, compute mean of all numeric features
correlation = ds.group_by('label').agg([
    pl.col(col).mean()
    for col in ds.columns[5:]
])

In [15]:
correlation.to_pandas()

,label,momentum_6m,momentum_3m,momentum_1m,above_200ma,pct_below_52w_high,rsi_14,volume_trend_20d,spy_above_200ma,spy_momentum_6m,gross_margin,net_margin,revenue_growth,net_income_trend,asset_liability_ratio,operating_cf_trend
0,1,23.202463,15.457208,11.391666,0.986123,-8.631737,68.697776,242632.853929,0.859497,7.892895,0.457066,-0.021853,123.023271,0.721888,2.321999,0.622302
1,0,20.665180,14.050426,10.395015,0.992512,NaN,69.751717,236453.716915,0.835642,7.010055,0.456513,-0.063893,104.310312,0.733649,2.122806,0.663250


In [19]:
ds.group_by('label').agg([
    pl.col('pct_below_52w_high').mean(),
    pl.col('pct_below_52w_high').null_count()
])

DuplicateError: column with name 'pct_below_52w_high' has more than one occurrence

Resolved plan until failure:

	---> FAILED HERE RESOLVING 'group_by' <---
DF ["ticker", "signal_date", "label", "alpha", ...]; PROJECT */20 COLUMNS

This error occurred with the following context stack:
	[1] 'group_by'


In [21]:
import polars as pl
import numpy as np
from scipy import stats

ds = pl.read_parquet('data/ml_labels.parquet')
labels = ds['label'].to_numpy()

features = [
    'momentum_6m', 'momentum_3m', 'momentum_1m', 'above_200ma',
    'pct_below_52w_high', 'rsi_14', 'spy_above_200ma', 'spy_momentum_6m',
    'gross_margin', 'net_margin', 'revenue_growth',
    'net_income_trend', 'asset_liability_ratio', 'operating_cf_trend', 'score'
]

results = []
for feat in features:
    vals = ds[feat].to_numpy()
    mask = ~np.isnan(vals.astype(float))
    if mask.sum() < 100:
        continue
    corr, pval = stats.pointbiserialr(labels[mask], vals[mask])
    results.append((feat, round(corr, 4), round(pval, 4)))

results.sort(key=lambda x: abs(x[1]), reverse=True)
print(f"{'Feature':<25} {'Correlation':>12} {'P-value':>10}")
print("-" * 50)
for feat, corr, pval in results:
    sig = "✓" if pval < 0.05 else " "
    print(f"{feat:<25} {corr:>12.4f} {pval:>10.4f}  {sig}")

Feature                    Correlation    P-value
--------------------------------------------------
pct_below_52w_high             -0.0875     0.0000  ✓
revenue_growth                  0.0544     0.0017  ✓
momentum_6m                     0.0520     0.0013  ✓
spy_momentum_6m                 0.0476     0.0032  ✓
score                           0.0464     0.0041  ✓
asset_liability_ratio           0.0431     0.0446  ✓
momentum_1m                     0.0419     0.0096  ✓
operating_cf_trend             -0.0390     0.2275   
momentum_3m                     0.0356     0.0277  ✓
rsi_14                         -0.0340     0.0354  ✓
above_200ma                    -0.0304     0.0605   
spy_above_200ma                 0.0301     0.0629   
net_income_trend               -0.0121     0.4843   
net_margin                      0.0052     0.7636   
gross_margin                    0.0008     0.9689   


In [24]:
import polars as pl

ds = pl.read_parquet('data/ml_labels.parquet')

ds = ds.drop(['gross_margin', 'net_margin', 'net_income_trend', 'operating_cf_trend'])

ds.write_parquet('data/ml_labels.parquet')
print(f"Final dataset: {ds.shape}")
print(f"Columns: {ds.columns}")

Final dataset: (3824, 16)
Columns: ['ticker', 'signal_date', 'label', 'alpha', 'score', 'momentum_6m', 'momentum_3m', 'momentum_1m', 'above_200ma', 'pct_below_52w_high', 'rsi_14', 'volume_trend_20d', 'spy_above_200ma', 'spy_momentum_6m', 'revenue_growth', 'asset_liability_ratio']


In [25]:
# Drop 100% null columns and identifiers not used as features
DROP_COLS = [
    'asset_to_mcap_ratio',  # 100% null
    'fcf_margin',           # 100% null
    'fcf_yield',            # 100% null
    'industry',             # 100% null
    'margin_trajectory',    # 100% null
    'market_cap',           # 100% null
    'revenue_growth_ttm',   # 100% null
    'sector',               # 100% null
    'trailing_pe',          # 100% null
    'close',                # raw price — not a feature
    'ma_200',               # raw MA level — not a feature
    'high_52w',             # raw price level — not a feature
]

In [28]:
import polars as pl
import numpy as np
from scipy import stats

ds = pl.read_parquet('data/ml_labels.parquet')

# Drop useless columns
drop = [c for c in DROP_COLS if c in ds.columns]
ds = ds.drop(drop)

# Save cleaned version
ds.write_parquet('data/ml_labels.parquet')
print(f"Cleaned shape: {ds.shape}")
print(f"Remaining columns: {ds.columns}")

Cleaned shape: (3879, 37)
Remaining columns: ['ticker', 'signal_date', 'label', 'alpha', 'score', 'above_200ma', 'pct_below_52w_high', 'in_dip', 'rsi_14', 'momentum_6m', 'momentum_3m', 'momentum_1m', 'momentum_1w', 'volume_trend_50d', 'revenue_consistency', 'revenue_growth_yoy', 'revenue_trajectory', 'net_income_improving', 'asset_liability_ratio_improving', 'asset_growth', 'liability_growth', 'cashflow_improving', 'gross_margin_latest', 'gross_margin_avg', 'margin_recent_avg', 'margin_older_avg', 'margin_stable_or_improving', 'market_cap_bucket', 'stock_vol_20d', 'stock_vol_63d', 'vol_ratio', 'market_vol_20d', 'spy_momentum_3m', 'spy_momentum_1m', 'market_drawdown', 'month', 'year_normalized']


In [29]:
labels = ds['label'].to_numpy()

skip = ['ticker', 'signal_date', 'label', 'alpha']
features = [c for c in ds.columns if c not in skip]

results = []
for feat in features:
    vals = ds[feat].cast(pl.Float64, strict=False).to_numpy().astype(float)
    mask = ~np.isnan(vals)
    if mask.sum() < 100:
        continue
    corr, pval = stats.pointbiserialr(labels[mask], vals[mask])
    results.append((feat, round(corr, 4), round(pval, 4), mask.sum()))

results.sort(key=lambda x: abs(x[1]), reverse=True)
print(f"\n{'Feature':<40} {'Corr':>8} {'P-val':>8} {'N':>6}")
print("-" * 65)
for feat, corr, pval, n in results:
    sig = "✓" if pval < 0.05 else " "
    print(f"{feat:<40} {corr:>8.4f} {pval:>8.4f} {n:>6}  {sig}")


Feature                                      Corr    P-val      N
-----------------------------------------------------------------
pct_below_52w_high                        -0.0876   0.0000   3819  ✓
market_cap_bucket                             nan      nan   3373   
stock_vol_63d                              0.1222   0.0000   3879  ✓
stock_vol_20d                              0.0773   0.0000   3879  ✓
in_dip                                     0.0575   0.0003   3879  ✓
momentum_6m                                0.0547   0.0007   3879  ✓
vol_ratio                                 -0.0520   0.0012   3879  ✓
score                                      0.0469   0.0035   3879  ✓
market_drawdown                            0.0451   0.0053   3824  ✓
momentum_1m                                0.0434   0.0069   3879  ✓
asset_liability_ratio_improving           -0.0431   0.0474   2114  ✓
above_200ma                               -0.0418   0.0092   3879  ✓
spy_momentum_1m                        

In [31]:
import polars as pl

ds = pl.read_parquet('data/ml_labels.parquet')

DROP = [
    'market_vol_20d', 'revenue_trajectory', 'margin_older_avg',
    'liability_growth', 'revenue_growth_yoy', 'asset_growth',
    'net_income_improving', 'year_normalized', 'gross_margin_avg',
    'margin_recent_avg', 'gross_margin_latest', 'volume_trend_50d',
    'cashflow_improving', 'revenue_consistency', 'margin_stable_or_improving',
    'market_cap_bucket', 'month'
]

drop = [c for c in DROP if c in ds.columns]
ds = ds.drop(drop)

ds.write_parquet('data/ml_labels.parquet')
print(f"Final dataset: {ds.shape}")
print(f"Features: {[c for c in ds.columns if c not in ['ticker', 'signal_date', 'label', 'alpha']]}")

Final dataset: (3879, 20)
Features: ['score', 'above_200ma', 'pct_below_52w_high', 'in_dip', 'rsi_14', 'momentum_6m', 'momentum_3m', 'momentum_1m', 'momentum_1w', 'asset_liability_ratio_improving', 'stock_vol_20d', 'stock_vol_63d', 'vol_ratio', 'spy_momentum_3m', 'spy_momentum_1m', 'market_drawdown']


In [32]:
import polars as pl

ds = pl.read_parquet('data/ml_labels.parquet')
trades = pl.read_parquet('/home/l/projects/trading-system/backtest/trades_A_6M.parquet')
prices = pl.read_parquet('/home/l/projects/trading-system/data/cache/prices.parquet')

# Pick 5 random signals and manually verify vol was computed correctly
sample = ds.sample(5, seed=42).select(['ticker', 'signal_date', 'stock_vol_63d'])

for row in sample.iter_rows(named=True):
    ticker = row['ticker']
    signal_date = row['signal_date']
    
    # Get prices strictly before signal date
    px = prices.filter(
        (pl.col('ticker') == ticker) &
        (pl.col('date') < signal_date)
    ).sort('date')
    
    # Manually compute vol_63d
    if len(px) >= 64:
        close = px['close'].to_numpy()
        returns = (close[1:] / close[:-1] - 1)
        manual_vol = float(returns[-63:].std())
    else:
        manual_vol = None
    
    stored_vol = row['stock_vol_63d']
    match = abs(manual_vol - stored_vol) < 0.0001 if manual_vol else False
    print(f"{ticker} {signal_date.date()} — stored: {stored_vol:.6f} manual: {manual_vol:.6f} match: {match}")

TPL 2022-12-19 — stored: 0.030900 manual: 0.030900 match: True
SW 2015-08-24 — stored: 0.011092 manual: 0.011092 match: True
BKR 2025-07-28 — stored: 0.019913 manual: 0.019913 match: True
CPRT 2021-03-22 — stored: 0.018184 manual: 0.018184 match: True
AIV 2022-08-15 — stored: 0.022135 manual: 0.022135 match: True


In [33]:
funds = pl.read_parquet('/home/l/projects/trading-system/data/cache/fundamentals_edgar.parquet')

# Pick 5 signals that have fundamental features
sample = ds.drop_nulls(subset=['asset_liability_ratio_improving']).sample(5, seed=42)

for row in sample.iter_rows(named=True):
    ticker = row['ticker']
    signal_date = row['signal_date']
    signal_date_str = signal_date.strftime('%Y-%m-%d')
    
    # Get all filings for this ticker
    filings = funds.filter(pl.col('ticker') == ticker).sort('filed')
    
    # Check if any filing used has a filed date AFTER signal date
    future_filings = filings.filter(pl.col('filed') > signal_date_str)
    pit_filings = filings.filter(pl.col('filed') <= signal_date_str)
    
    print(f"{ticker} {signal_date.date()} — PIT filings: {len(pit_filings)} Future filings: {len(future_filings)}")

JEF 2023-07-03 — PIT filings: 50 Future filings: 2
BX 2016-11-21 — PIT filings: 15 Future filings: 45
GL 2025-07-28 — PIT filings: 63 Future filings: 7
GT 2022-01-18 — PIT filings: 45 Future filings: 22
AFL 2023-04-10 — PIT filings: 47 Future filings: 17


In [34]:
import numpy as np
from scipy import stats

# Join raw return_pct onto dataset
ds_check = ds.join(
    trades.select(['ticker', 'signal_date', 'return_pct']),
    on=['ticker', 'signal_date']
)

raw_returns = ds_check['return_pct'].to_numpy()

features = [c for c in ds.columns 
            if c not in ['ticker', 'signal_date', 'label', 'alpha']]

print(f"{'Feature':<40} {'Corr with return_pct':>20} {'Corr with label':>16}")
print("-" * 80)
labels = ds['label'].to_numpy()

for feat in features:
    vals = ds_check[feat].cast(pl.Float64, strict=False).to_numpy().astype(float)
    mask = ~np.isnan(vals)
    if mask.sum() < 100:
        continue
    corr_return, _ = stats.pearsonr(raw_returns[mask], vals[mask])
    corr_label, _  = stats.pointbiserialr(labels[mask], vals[mask])
    flag = " ⚠️" if abs(corr_return) > 0.3 else ""
    print(f"{feat:<40} {corr_return:>20.4f} {corr_label:>16.4f}{flag}")

Feature                                  Corr with return_pct  Corr with label
--------------------------------------------------------------------------------
score                                                  0.0380           0.0469
above_200ma                                           -0.0586          -0.0418
pct_below_52w_high                                    -0.1133          -0.0876
in_dip                                                 0.0788           0.0575
rsi_14                                                -0.0369          -0.0322
momentum_6m                                            0.0310           0.0547
momentum_3m                                            0.0106           0.0285
momentum_1m                                            0.0472           0.0434
momentum_1w                                            0.0465           0.0356
asset_liability_ratio_improving                       -0.0177          -0.0431
stock_vol_20d                                     

In [35]:
for row in sample.iter_rows(named=True):
    ticker = row['ticker']
    signal_date = row['signal_date']
    signal_date_str = signal_date.strftime('%Y-%m-%d')
    
    # What filings would fundamental_features_edgar actually use?
    pit = funds.filter(
        (pl.col('ticker') == ticker) &
        (pl.col('filed') <= signal_date_str)
    ).sort('period_end', descending=True).head(4)
    
    print(f"\n{ticker} {signal_date.date()} — 4 most recent PIT filings:")
    print(pit.select(['ticker', 'period_end', 'filed']))


JEF 2023-07-03 — 4 most recent PIT filings:
shape: (4, 3)
┌────────┬────────────┬────────────┐
│ ticker ┆ period_end ┆ filed      │
│ ---    ┆ ---        ┆ ---        │
│ str    ┆ str        ┆ str        │
╞════════╪════════════╪════════════╡
│ JEF    ┆ 2020-11-30 ┆ 2023-01-27 │
│ JEF    ┆ 2020-08-31 ┆ 2021-10-08 │
│ JEF    ┆ 2020-05-31 ┆ 2021-07-09 │
│ JEF    ┆ 2020-02-29 ┆ 2021-04-08 │
└────────┴────────────┴────────────┘

BX 2016-11-21 — 4 most recent PIT filings:
shape: (4, 3)
┌────────┬────────────┬────────────┐
│ ticker ┆ period_end ┆ filed      │
│ ---    ┆ ---        ┆ ---        │
│ str    ┆ str        ┆ str        │
╞════════╪════════════╪════════════╡
│ BX     ┆ 2014-09-30 ┆ 2016-02-26 │
│ BX     ┆ 2014-06-30 ┆ 2016-02-26 │
│ BX     ┆ 2014-03-31 ┆ 2016-02-26 │
│ BX     ┆ 2013-12-31 ┆ 2016-02-26 │
└────────┴────────────┴────────────┘

GL 2025-07-28 — 4 most recent PIT filings:
shape: (4, 3)
┌────────┬────────────┬────────────┐
│ ticker ┆ period_end ┆ filed      │
│ ---    ┆ 

In [38]:
import polars as pl

ds = pl.read_parquet('data/ml_labels.parquet')
ds = ds.drop('asset_liability_ratio_improving')
ds.write_parquet('data/ml_labels.parquet')

print(f"Final dataset: {ds.shape}")
print(f"\nFeatures:")
features = [c for c in ds.columns if c not in ['ticker', 'signal_date', 'label', 'alpha']]
for f in features:
    null_pct = ds[f].null_count() / len(ds) * 100
    print(f"  {f:<35} nulls: {null_pct:.1f}%")

print(f"\nLabel distribution:")
print(ds['label'].value_counts())

print(f"\nAlpha distribution:")
print(ds['alpha'].describe())

Final dataset: (3879, 19)

Features:
  score                               nulls: 0.0%
  above_200ma                         nulls: 0.0%
  pct_below_52w_high                  nulls: 1.5%
  in_dip                              nulls: 0.0%
  rsi_14                              nulls: 0.0%
  momentum_6m                         nulls: 0.0%
  momentum_3m                         nulls: 0.0%
  momentum_1m                         nulls: 0.0%
  momentum_1w                         nulls: 0.0%
  stock_vol_20d                       nulls: 0.0%
  stock_vol_63d                       nulls: 0.0%
  vol_ratio                           nulls: 0.0%
  spy_momentum_3m                     nulls: 1.4%
  spy_momentum_1m                     nulls: 1.4%
  market_drawdown                     nulls: 1.4%

Label distribution:
shape: (2, 2)
┌───────┬───────┐
│ label ┆ count │
│ ---   ┆ ---   │
│ i64   ┆ u32   │
╞═══════╪═══════╡
│ 1     ┆ 1172  │
│ 0     ┆ 2707  │
└───────┴───────┘

Alpha distribution:
shape: (9, 2)

In [39]:
import polars as pl

ds = pl.read_parquet('data/ml_labels.parquet')

print("Alpha by label:")
print(ds.group_by('label').agg([
    pl.col('alpha').mean().alias('mean_alpha'),
    pl.col('alpha').median().alias('median_alpha'),
    pl.col('alpha').std().alias('std_alpha'),
    pl.col('alpha').min().alias('min_alpha'),
    pl.col('alpha').max().alias('max_alpha'),
    pl.len().alias('count')
]).sort('label'))

# Alpha percentiles for losing signals
print("\nAlpha percentiles for label=0 signals:")
label0 = ds.filter(pl.col('label') == 0)['alpha']
for p in [10, 25, 50, 75, 90]:
    print(f"  P{p}: {label0.quantile(p/100):.1f}%")

print("\nAlpha percentiles for label=1 signals:")
label1 = ds.filter(pl.col('label') == 1)['alpha']
for p in [10, 25, 50, 75, 90]:
    print(f"  P{p}: {label1.quantile(p/100):.1f}%")

Alpha by label:
shape: (2, 7)
┌───────┬────────────┬──────────────┬───────────┬────────────┬────────────┬───────┐
│ label ┆ mean_alpha ┆ median_alpha ┆ std_alpha ┆ min_alpha  ┆ max_alpha  ┆ count │
│ ---   ┆ ---        ┆ ---          ┆ ---       ┆ ---        ┆ ---        ┆ ---   │
│ i64   ┆ f64        ┆ f64          ┆ f64       ┆ f64        ┆ f64        ┆ u32   │
╞═══════╪════════════╪══════════════╪═══════════╪════════════╪════════════╪═══════╡
│ 0     ┆ -8.859994  ┆ -6.317001    ┆ 13.538691 ┆ -78.233635 ┆ 9.966688   ┆ 2707  │
│ 1     ┆ 29.754791  ┆ 21.248093    ┆ 28.369792 ┆ 10.053809  ┆ 348.559396 ┆ 1172  │
└───────┴────────────┴──────────────┴───────────┴────────────┴────────────┴───────┘

Alpha percentiles for label=0 signals:
  P10: -27.6%
  P25: -16.7%
  P50: -6.3%
  P75: 1.6%
  P90: 6.5%

Alpha percentiles for label=1 signals:
  P10: 12.0%
  P25: 15.1%
  P50: 21.3%
  P75: 34.0%
  P90: 54.9%


In [40]:
import polars as pl

ds = pl.read_parquet('data/ml_labels.parquet')
trades = pl.read_parquet('/home/l/projects/trading-system/backtest/trades_A_6M.parquet')

print("=== BASELINE A_6M PERFORMANCE (no ML filter) ===")
print(f"Total signals: {len(trades)}")
print(f"Mean return per trade: {trades['return_pct'].mean():.2f}%")
print(f"Mean benchmark return: {trades['benchmark_return'].mean():.2f}%")
print(f"Mean alpha per trade: {(trades['return_pct'] - trades['benchmark_return']).mean():.2f}%")
print(f"Win rate vs SPY: {trades['outperformed'].mean()*100:.1f}%")
print(f"Signals per year (approx): {len(trades)/15:.0f}")

print("\n=== WHAT ML FILTER COULD THEORETICALLY ADD ===")
label1 = ds.filter(pl.col('label') == 1)
label0 = ds.filter(pl.col('label') == 0)
print(f"Label=1 signals mean alpha: {label1['alpha'].mean():.2f}%")
print(f"Label=0 signals mean alpha: {label0['alpha'].mean():.2f}%")
print(f"Alpha gap between groups: {label1['alpha'].mean() - label0['alpha'].mean():.2f}%")
print(f"If model perfectly filtered label=0: mean alpha would be {label1['alpha'].mean():.2f}%")

=== BASELINE A_6M PERFORMANCE (no ML filter) ===
Total signals: 3882
Mean return per trade: 9.57%
Mean benchmark return: 6.76%
Mean alpha per trade: 2.81%
Win rate vs SPY: 51.0%
Signals per year (approx): 259

=== WHAT ML FILTER COULD THEORETICALLY ADD ===
Label=1 signals mean alpha: 29.75%
Label=0 signals mean alpha: -8.86%
Alpha gap between groups: 38.61%
If model perfectly filtered label=0: mean alpha would be 29.75%
